In [12]:
# Program to find origin, provider, prefix, its length and version from raw data generated using bgpreader as 
# bgpreader -t ribs -w 1485907200,1485907200 -A _32787_ >> /home/shyam/jupy/ddos_scrubber/data/raw_as32787_01_feb_2017.txt
# Use in aruba
# Use python multicore programming feature

year = "2017" 
month = "feb"
asn = "32787" # Scrubber asn

import pandas as pd
import csv
from concurrent.futures import ProcessPoolExecutor

def process_chunk(lines):
    """Process a chunk of lines and extract prefix, AS path, and origin AS."""
    chunk_data = []
    for line in lines:
        fields = line.strip().split('|')
        if len(fields) > 12 and fields[1] == "R":
            try:
               # Extract prefix, AS path, and origin AS
                prefix = fields[9]
                as_path = fields[11].split()


                # Find provider here checking AS repetetitions and the same organization owning multiple ASes
#                 provider = find_immediate_provider(as_path)
                
                provider = as_path[-2] if len(as_path) > 1 else None  # The ASN before the origin AS
                origin_as = as_path[-1] if as_path else None
                
                pfx_len = int(prefix.split('/')[1]) if '/' in prefix else None
                ip_version = "IPv6" if ':' in prefix else "IPv4"

                # Append data to the chunk
                chunk_data.append([prefix, ' '.join(as_path), origin_as, provider, pfx_len, ip_version])
            
            except IndexError:
                continue
    return chunk_data

def process_route_data_parallel(file_path, output_file, num_workers=8, chunk_size=100000):
    with open(file_path, 'r') as file:
        lines = []
        
        # Initialize CSV output
        with open(output_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['prefix', 'as_path', 'origin_as', 'provider', 'pfx_len', 'ip_version'])

        # Process the file in chunks with multiple workers
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            futures = []
            for line in file:
                lines.append(line)
                
                # When lines reach chunk size, process them
                if len(lines) >= chunk_size:
                    futures.append(executor.submit(process_chunk, lines))
                    lines = []

            # Process any remaining lines after the loop
            if lines:
                futures.append(executor.submit(process_chunk, lines))

            # Collect results from all futures and write to CSV
            for future in futures:
                chunk_data = future.result()
                if chunk_data:
                    df = pd.DataFrame(chunk_data, columns=['prefix', 'as_path', 'origin_as', 'provider', 'pfx_len', 'ip_version'])
                    df.to_csv(output_file, mode='a', header=False, index=False)

    print(f"Data successfully saved to {output_file}")

    
# Loop through the txt files that contain RIB snapshots of every month   
# Example usage
file_path = '/home/shyam/jupy/ddos_scrubber/data/raw_as'+asn+'_01_jan_'+year+'.txt'
output_file = '/home/shyam/jupy/ddos_scrubber/data/optimized_raw_as'+asn+'_01_jan_'+year+'.csv'
process_route_data_parallel(file_path, output_file)


Data successfully saved to /home/shyam/jupy/ddos_scrubber/data/optimized_raw_as13335_01_jan_2024.csv


In [58]:
# t = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/optimized_raw_as19905_01_jan_2024.csv")
# print(len(t))
# # t.loc[t["provider"] != 12008]
# siblings = [397221,399169,399168,397231,399177,399164,397218,397222,399173,399157,399151,397217,399163,
#             397223,397235,399171,397230,399170,397214,19911,399166,399172,399158,397213,399174,397239,399178,
#             399180,22701,397215,397233,399167,399176,397238,397219,399155,397216,399175,397237,399154,397240,
#             397224,399159,397225,397227,397243,12008,397234,399152,397229,399165,397220,397226,397228,399179,
#             397236,399162,399153,399161,397242,397241,399160,397232,399156]

# # Filter rows where 'asns' is NOT in L2
# filtered_df = t[t['origin_as'].isin(siblings)]

# print(filtered_df)

229143
                 prefix                                      as_path  \
996     156.154.94.0/24  6423 2044 174 1299 22822 19905 36236 399153   
997     156.154.94.0/24            1798 6461 6453 19905 36236 399153   
2381    156.154.94.0/24                20080 2914 19905 36236 399157   
2382    156.154.94.0/24               199524 2914 19905 36236 399157   
2383    156.154.94.0/24                52320 2914 19905 36236 399157   
...                 ...                                          ...   
225237  156.154.94.0/24         199524 37100 2914 19905 36236 399158   
225238  156.154.94.0/24                 3257 6453 19905 36236 399157   
225239  156.154.94.0/24         327804 37271 2914 19905 36236 399158   
225240  156.154.94.0/24     327960 174 1299 22822 19905 36236 399158   
227587  156.154.94.0/24           29680 3257 6453 19905 36236 399157   

        origin_as  provider  pfx_len ip_version  
996        399153     36236       24       IPv4  
997        399153     36236 

In [13]:
# Read csv file and remove duplicate rows
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/optimized_raw_as'+asn+'_01_jan_'+year+'.csv', low_memory=False)
 
# Remove duplicate rows in dataframe
df = df.drop_duplicates()

# Write unique rows into a file
df.to_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as'+asn+'_01_jan_'+year+'.csv', index=False)
df

,prefix,as_path,origin_as,provider,pfx_len,ip_version
0,1.0.0.0/24,41666 13335,13335,41666.0,24,IPv4
1,1.0.0.0/24,208594 12735 13335,13335,12735.0,24,IPv4
2,1.0.0.0/24,207934 13335,13335,207934.0,24,IPv4
3,1.0.0.0/24,19529 395348 6939 13335,13335,6939.0,24,IPv4
4,1.0.0.0/24,49544 3356 6762 13335,13335,6762.0,24,IPv4
...,...,...,...,...,...,...
2804930,23.29.145.0/24,3741 13335 394303,394303,13335.0,24,IPv4
2804931,23.29.145.0/24,3561 209 3356 13335 394303,394303,13335.0,24,IPv4
2804933,23.29.145.0/24,34224 13335 394303,394303,13335.0,24,IPv4
2804935,23.29.145.0/24,20130 6939 13335 394303,394303,13335.0,24,IPv4


In [14]:
# Remove records containing AS set on an AS path
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as'+asn+'_01_jan_'+year+'.csv', low_memory=False)
print("Records before %s." %len(df)) 

print("Removing AS set in an AS path")
# Find rows containing '{}' (set origins)
mask = df['as_path'].str.contains(r'\{.*\}')
rows_with_origin_set = df[mask]

# Count rows with set origin
count_set_origin = rows_with_origin_set.shape[0]

# Remove rows with set origin from the DataFrame
df_cleaned = df[~mask]

# Remove rows with set origin from the DataFrame
df_cleaned = df[~mask]

df_cleaned.to_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as'+asn+'_01_jan_'+year+'.csv', index = False)
print("Number of rows with set origin:", count_set_origin)
print("DataFrame after removal:")
print("Records after %s." %len(df_cleaned)) 
print("%s number of records were removed that contains AS set in AS paths." %(len(df) - len(df_cleaned)))


Records before 1991544.
Removing AS set in an AS path
Number of rows with set origin: 0
DataFrame after removal:
Records after 1991544.
0 number of records were removed that contains AS set in AS paths.


In [15]:
import pandas as pd
year = "2024"
asn = "13335"

df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as'+asn+'_01_jan_'+year+'.csv', low_memory=False)
df

,prefix,as_path,origin_as,provider,pfx_len,ip_version
0,1.0.0.0/24,41666 13335,13335,41666.0,24,IPv4
1,1.0.0.0/24,208594 12735 13335,13335,12735.0,24,IPv4
2,1.0.0.0/24,207934 13335,13335,207934.0,24,IPv4
3,1.0.0.0/24,19529 395348 6939 13335,13335,6939.0,24,IPv4
4,1.0.0.0/24,49544 3356 6762 13335,13335,6762.0,24,IPv4
...,...,...,...,...,...,...
1991539,23.29.145.0/24,3741 13335 394303,394303,13335.0,24,IPv4
1991540,23.29.145.0/24,3561 209 3356 13335 394303,394303,13335.0,24,IPv4
1991541,23.29.145.0/24,34224 13335 394303,394303,13335.0,24,IPv4
1991542,23.29.145.0/24,20130 6939 13335 394303,394303,13335.0,24,IPv4


In [2]:
import re
from collections import defaultdict

def count_prefix_lengths_and_versions(prefixes):
    # Patterns to match IPv4 and IPv6
    ipv4_pattern = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")
    ipv6_pattern = re.compile(r"^(?:[a-fA-F0-9]{1,4}:){1,7}[a-fA-F0-9]{1,4}$|^(?:[a-fA-F0-9]{0,4}:){1,7}:$|^::(?:[a-fA-F0-9]{1,4}:){0,6}[a-fA-F0-9]{1,4}$")
    
    # Dictionary to hold counts based on IP version and prefix length
    counts = defaultdict(lambda: defaultdict(int))

    for prefix in prefixes:
        # Split into IP address and prefix length
        ip, length = prefix.split('/')
        
        # Detect IP version and update counts
        if ipv4_pattern.match(ip):
            ip_version = "IPv4"
        elif ipv6_pattern.match(ip):
            ip_version = "IPv6"
        else:
            continue  # Ignore invalid IPs if any

        # Update counts
        counts[ip_version][length] += 1

    return counts

In [17]:
# Find records that have origin differnet from AS198949 and their siblings; and contain AS198949 as 
# the second last hop AS on their AS paths

print("%s number of records in RIB file that have AS%s on their AS Paths" %(len(df), asn))
# Find number of unique prefixes that contain AS198949 as origin
condition = (df['origin_as'] == int(asn)) 
df2 = df.loc[condition]
print("%s number of prefixes have origin AS%s" %(len(df2), asn))

# Find number of unique prefixes that do not contain AS198949 as well as its siblings as origin
# Siblings are found using ASRank API for the date 2024-01-01
siblings = [394536, 395747, 14789, 13335]


condition = (~df['origin_as'].isin (siblings)) 
df2 = df.loc[condition]
print("%s number of prefixes do not have origin AS%s and its siblings" %(len(df2), asn))

# Prefixes containing scrubber asn as the second last hop AS in AS path
provider_scrubber_records = df2.loc[df['provider'] == int(asn)]
print("%s number of prefixes contain AS%s as a provider(second last hop in AS path)" %(len(provider_scrubber_records), asn))

# Save confirmed_costumers1 to a csv file
provider_scrubber_records.to_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv", index=False)
# Unique origin ASes are actually CONFIRMED customers that have their second last hop AS as AS198949
unique_origin_ases = provider_scrubber_records['origin_as'].unique()
print("%s number of unique customer ASNs of AS%s" %(len(unique_origin_ases), asn))

unique_prefixes = provider_scrubber_records['prefix'].unique()
print("%s number of unique prefixes that contain AS%s as provider" %(len(unique_prefixes), asn))

# # Find prefixes sizes, their numbers and IP versions
result = count_prefix_lengths_and_versions(unique_prefixes)

# Print results
for version, lengths in result.items():
    print(f"{version}:")
    for length, count in lengths.items():
        print(f"  /{length}: {count} prefixes")

1991544 number of records in RIB file that have AS13335 on their AS Paths
1008942 number of prefixes have origin AS13335
980857 number of prefixes do not have origin AS13335 and its siblings
980151 number of prefixes contain AS13335 as a provider(second last hop in AS path)
632 number of unique customer ASNs of AS13335
3321 number of unique prefixes that contain AS13335 as provider
IPv4:
  /24: 2372 prefixes
  /22: 247 prefixes
  /23: 424 prefixes
  /21: 84 prefixes
  /19: 50 prefixes
  /17: 13 prefixes
  /20: 45 prefixes
  /16: 11 prefixes
  /13: 1 prefixes
  /18: 17 prefixes
  /14: 1 prefixes
IPv6:
  /48: 51 prefixes
  /40: 1 prefixes
  /47: 1 prefixes
  /32: 3 prefixes


In [18]:
# Look for the cases where records contain the second last hop AS other than AS198949 => Semi-confirmed customers
# Probable reasons: 1. AS path prepending 2. Sibling ASes
# 1. AS path prepending
import pandas as pd
# df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_as198949_01_jan_2024.csv", low_memory=False)
provider_not_scrubber = df2.loc[df['provider'] != int(asn)]
provider_not_scrubber.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+".csv", index=False)
print("%s number of prefixes do not contain AS%s as a provider (second last hop in AS path)." %(len(provider_not_scrubber), asn))

706 number of prefixes do not contain AS13335 as a provider (second last hop in AS path).


In [1]:
# CASE #1: AS path prepending for the records containing the second last hop AS other than AS198949 => Semi-confirmed customers 
import pandas as pd

# Define a function to determine path prepending and new_provider
def determine_path_prepending_and_new_provider(row):
   
    # Ensure as_path is a string and split into a list of ASNs
    as_path = list(map(int, str(row['as_path']).split()))
    provider = None # Default value set
    if len(as_path) < 2:
        provider = None  # Not enough ASNs in path to determine provider

    origin_as = as_path[-1]  # The last ASN is the origin ASN
    
    path_prepending = 1 if as_path.count(origin_as) > 1 else 0

#     provider = as_path[-2]  # If all checks fail, return the second ASN by default
    
    # Check for sequentially repeated ASNs
    repeated_asn = None
    for i in range(len(as_path) - 1, 0, -1):
        if as_path[i-1] == as_path[i]:
            repeated_asn = as_path[i]
        else:
            # If we find an ASN that is not the same as the repeated ASN,
            # and we have found a repeated ASN, return the one before the repeated ASN
            if repeated_asn is not None:
                provider = as_path[i-1]  # This is the upstream provider
            break
    
    return pd.Series([path_prepending, provider])

df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+".csv")

# Apply the function to each row
df[['path_prepending', 'new_provider']] = df.apply(determine_path_prepending_and_new_provider, axis=1)

# Print the updated DataFrame to verify results
df.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+"_v2.csv") 

# Find the records of AS path prepending
condition = (df['path_prepending'] == 1) & (df['new_provider'] == int(asn))
provider_scrubber_as_path_prepend = df.loc[condition]
print("%s number of records have AS path prepended and contain AS%s as a provider(not a second last hop in AS path though)" %(len(provider_scrubber_as_path_prepend), asn))
provider_scrubber_as_path_prepend.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as"+asn+"_path_prepend_01_jan_"+year+".csv")

NameError: name 'asn' is not defined

In [ ]:
# CASE #2: Check if there exists sibling ASes of the origin AS => Semi-confirmed customers 2 
# Do it in Aruba machine
# ssh -L 8900:localhost:8900 aruba-shyam 
 

# Then in your local browser go to: http://127.0.0.1:8900/lab/workspaces/ 

# Save your code under workspace/notebooks 

# http://127.0.0.1:8900/lab?token=f83423d33e90909dc6d48114425561335d5f07713b529c18 
        
        
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

# Function to determine sibling ASes and new provider
def determine_sibling_ases_and_new_provider(row):
    # Ensure as_path is a string and split into a list of ASNs
    as_path = list(map(int, str(row['as_path']).split()))
    
    # If as_path is too short to determine provider, return default values
    if len(as_path) < 2:
        return (0, None)  # No siblings, no provider

    origin_as = as_path[-1]  # The last ASN is the origin ASN
    provider = as_path[-2]  # Default provider is the second-to-last ASN
    
    # Default sibling count
    siblings = 0 
    
    # Retrieve sibling list from an API (or mock it here for testing)
    try:
        sibling_list = find_siblings(str(origin_as))
    except Exception as e:
        print(f"Error retrieving siblings: {e}")
        return (0, None)  # No siblings, no provider in case of API failure

    # Traverse the AS path to identify a non-sibling provider
    for i in range(len(as_path) - 1, 0, -1):
        if str(as_path[i-1]) not in sibling_list:
            provider = as_path[i-1]
            break

    # Determine if there are any siblings in the AS path
    as_path_str = list(map(str, as_path))  # Convert AS path to strings for comparison
    siblings = 1 if len(set(as_path_str) & set(sibling_list)) > 0 else 0

    return (siblings, provider)  # Return as tuple

# Function to process a chunk of the DataFrame
def process_chunk(chunk):
    # Apply the function and collect results
    results = chunk.apply(determine_sibling_ases_and_new_provider, axis=1)
    
    # Convert results to DataFrame with correct columns
    results_df = pd.DataFrame(results.tolist(), columns=['siblings', 'new_provider_sibling_check'], index=chunk.index)
    
    return results_df

# Load dataset and filter by specific criteria
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+"_v2.csv")
df = df.loc[df['path_prepending'] == 0]
print(f"{len(df)} records found with AS path prepending = 0 and provider not AS {asn})

# Define chunk size and split data into chunks for parallel processing
chunk_size = 1000  # Adjust based on available CPU cores and memory
chunks = [df[i:i + chunk_size] for i in range(0, len(df), chunk_size)]

# Set the number of workers (processors)
num_workers = 15  # Adjust this number based on your CPU

# Use ProcessPoolExecutor for parallel processing
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    # Map the function over each chunk and gather results
    results = list(executor.map(process_chunk, chunks))

# Combine processed chunks into a single DataFrame with reset index
processed_chunks = pd.concat(results)

# Assign processed data to the original DataFrame columns
df[['siblings', 'new_provider_sibling_check']] = processed_chunks

# Print final output or save to file
df.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"01_jan_"+year+"_v3.csv")

In [40]:
# Find the number of records containing siblings and not siblings. 

import pandas as pd
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+"_v3.csv")

# Confirmed customers (AS198949 comes immediately after the siblings) after sibling check
confirmed_customers_after_sibling_check = df.loc[(df['siblings'] == 1) & (df['new_provider_sibling_check'] == int(asn))]
print("%s number of records have sibling ASes and contain AS%s as a provider (not a second last hop in AS path though)" %(len(confirmed_customers_after_sibling_check), asn))

no_sibling_different_provider_records = df.loc[(df['siblings'] == 0) & (df['new_provider_sibling_check'] != int(asn))]
print("%s number of records do not have sibling ASes and contain another provider" %len(no_sibling_different_provider_records))
no_sibling_different_provider_records

1872 number of records have sibling ASes and contain AS32787 as a provider (not a second last hop in AS path though)
5055 number of records do not have sibling ASes and contain another provider


,Unnamed: 0.1,Unnamed: 0,prefix,as_path,origin_as,provider,pfx_len,ip_version,path_prepending,new_provider,siblings,new_provider_sibling_check
4,277,277,64.30.234.0/23,2914 32787 6623 6641,6641,6623.0,23,IPv4,0.0,NaN,0,6623
5,278,278,64.30.234.0/23,14061 6461 32787 6623 6641,6641,6623.0,23,IPv4,0.0,NaN,0,6623
6,279,279,64.30.234.0/23,19151 32787 6623 6641,6641,6623.0,23,IPv4,0.0,NaN,0,6623
7,280,280,64.30.234.0/23,7575 6461 32787 6623 6641,6641,6623.0,23,IPv4,0.0,NaN,0,6623
8,281,281,64.30.234.0/23,14907 32787 6623 6641,6641,6623.0,23,IPv4,0.0,NaN,0,6623
...,...,...,...,...,...,...,...,...,...,...,...,...
6925,36966,36966,203.26.51.0/24,43578 1299 32787 9667 23854,23854,9667.0,24,IPv4,0.0,NaN,0,9667
6926,36967,36967,203.26.177.0/24,29680 6461 32787 9667 23854,23854,9667.0,24,IPv4,0.0,NaN,0,9667
6927,36968,36968,203.26.177.0/24,43578 1299 32787 9667 23854,23854,9667.0,24,IPv4,0.0,NaN,0,9667
6928,36969,36969,203.31.75.0/24,29680 6461 32787 9667 23854,23854,9667.0,24,IPv4,0.0,NaN,0,9667


In [41]:
# Find customers of 198949 where it is not an immediate ASN e.g. 1798 6461 198949 43338 44815
# It might be used to validate our method of finding the number of customers with AS rank customer ASNs
import pandas as pd

# Main thing is to find the location of the scrubber in an AS path
# Function to extract ASes after 198949
def find_ases_after(as_path):
    ases = as_path.split()
    if asn in ases:
        idx = ases.index(asn)
        return ases[idx + 1:]
    return []

# Apply the function to the as_path column and get a unique list of ASes
customer_asns_not_sibling_not_immediate_provider = no_sibling_different_provider_records['as_path'].apply(find_ases_after).explode().unique().tolist()

print("%s ASes lie after AS%s, not as the second last hop in their AS paths."%(len(customer_asns_not_sibling_not_immediate_provider), asn))


18 ASes lie after AS32787, not as the second last hop in their AS paths.


In [3]:
# Compare confirmed_customers_after_sibling_check, path prepending with the confirmed customers
year = "2024"
asn = "32787"
import pandas as pd

# After checking second last hop AS as AS198949
confirmed_customers1 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")
confirmed_customers1_unique_origin_ases = confirmed_customers1['origin_as'].unique()
confirmed_customers1_unique_prefixes = confirmed_customers1['prefix'].unique()

# After checking path prepending
path_prepending = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as"+asn+"_path_prepend_01_jan_"+year+".csv")
path_prepending_unique_origin_ases = path_prepending['origin_as'].unique()
path_prepending_unique_prefixes = path_prepending['prefix'].unique()

ases_sometimes_prepend_not_prepend = set(confirmed_customers1_unique_origin_ases) & set(path_prepending_unique_origin_ases)
print("%s number of ASes sometimes prepend their AS numbers and sometimes not for their different prefixes: %s ." %(len(ases_sometimes_prepend_not_prepend), (ases_sometimes_prepend_not_prepend)))

confirmed_customers2 = list(set(path_prepending_unique_origin_ases)-set(confirmed_customers1_unique_origin_ases))
print("%s number of unique customer ASNs of AS%s that prepend their origin." %(len(confirmed_customers2), asn))

prefixes_sometimes_prepend_not_prepend = set(confirmed_customers1_unique_prefixes) & set(path_prepending_unique_prefixes)
print("%s number of prefixes are sometimes prepended by origin and sometimes not %s ." %(len(prefixes_sometimes_prepend_not_prepend), (prefixes_sometimes_prepend_not_prepend)))

new_prefixes_prepending = list(set(path_prepending_unique_prefixes)-set(confirmed_customers1_unique_prefixes))
print("%s number of unique prefixes that contain AS%s as provider after checking path prepending." %(len(new_prefixes_prepending), asn))

# # Find prefixes sizes, their numbers and IP versions
result = count_prefix_lengths_and_versions(new_prefixes_prepending)

# Print results
for version, lengths in result.items():
    print(f"{version}:")
    for length, count in lengths.items():
        print(f"  /{length}: {count} prefixes")
   


# After checking condition that AS198949 comes immediately after the siblings
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+"_v3.csv")
sibling_path = df.loc[(df['siblings'] == 1) & (df['new_provider_sibling_check'] == int(asn))]

sibling_path_unique_origin_ases = sibling_path['origin_as'].unique()
sibling_path_unique_prefixes = sibling_path['prefix'].unique()

confirmed_customers3 = list(set(sibling_path_unique_origin_ases)-set(confirmed_customers2)-set(confirmed_customers1_unique_origin_ases))
print("%s number of unique customer ASNs of AS%s that have siblings on their origin : %s." %(len(confirmed_customers3), asn, confirmed_customers3))

new_prefixes_siblings = list(set(sibling_path_unique_prefixes)- set(path_prepending_unique_prefixes) - set(confirmed_customers1_unique_prefixes))
print("%s number of unique prefixes that contain AS%sas provider after checking siblings." %(len(new_prefixes_siblings), asn))

# # Find prefixes sizes, their numbers and IP versions
result = count_prefix_lengths_and_versions(new_prefixes_siblings)

# Print results
for version, lengths in result.items():
    print(f"{version}:")
    for length, count in lengths.items():
        print(f"  /{length}: {count} prefixes")
   

18 number of ASes sometimes prepend their AS numbers and sometimes not for their different prefixes: {13888, 51776, 15810, 3, 6949, 45574, 11911, 11685, 63335, 15914, 54986, 48075, 46513, 40410, 7734, 41594, 19738, 25215} .
21 number of unique customer ASNs of AS32787 that prepend their origin.
9 number of prefixes are sometimes prepended by origin and sometimes not {'155.140.102.0/24', '89.107.176.0/24', '155.140.82.0/24', '159.50.124.0/24', '155.140.83.0/24', '155.140.89.0/24', '155.140.103.0/24', '185.24.7.0/24', '155.140.88.0/24'} .
224 number of unique prefixes that contain AS32787 as provider after checking path prepending.
IPv4:
  /24: 161 prefixes
  /23: 21 prefixes
  /21: 1 prefixes
  /22: 15 prefixes
  /19: 2 prefixes
  /16: 1 prefixes
  /17: 2 prefixes
IPv6:
  /48: 18 prefixes
  /33: 2 prefixes
  /34: 1 prefixes
6 number of unique customer ASNs of AS32787 that have siblings on their origin : [203584, 16838, 23784, 27533, 48536, 21342].
89 number of unique prefixes that conta

In [ ]:
# Required to validate our finding with AS rank api
# Some bugs here
# new_origin_ases_without_siblings_not_immediate_provider = list(set(confirmed_customers3_unique_origin_ases) | set(confirmed_customers2_unique_origin_ases)| set(confirmed_customers1_unique_origin_ases) | set(customer_asns_not_sibling_not_immediate_provider))
# print("%s number of unique customer ASNs of AS32787 that contains that AS not as the second last hop on their origin : %s." %(len(new_origin_ases_without_siblings_not_immediate_provider), new_origin_ases_without_siblings_not_immediate_provider))




In [ ]:
# TO COMPLETE: Find the final list of customer ASNs and total prefixes
import pandas as pd
# df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as32787_01_jan_2024_v3.csv")
# df = df.loc[df["origin_as"] == 27533]
# unique_prefix = df["prefix"].unique()
# unique_prefix

# Find the number of prefixes originated by that ASN on that day looking into RIB file.
# The objective is to find what portion of prefixes are protected for each customer. 

# Characetrize the customers: For example: Bank customers protect all of their prefixes

# Do 

In [ ]:
# Program to find siblings of an ASN based on CAIDA AS2Org data
# Read line from 95721 from the file /h # format:aut|changed|aut_name|org_id|opaque_id|source
import gzip
import json

# Function to find siblings of a given ASN in a .jsonl.gz file, starting from a specific line
def find_siblings(asn_to_find):
    # Create a dictionary to group ASNs by organizationId
    org_id_map = {}
    
    file_path = '/home/shyam/jupy/ddos_scrubber/data/20241001.as-org2info.jsonl.gz'

    start_line = 95721
        
    # Open and read the .jsonl.gz file
    with gzip.open(file_path, 'rt') as f:  # 'rt' is for reading text mode
        for current_line, line in enumerate(f):
            if current_line < start_line:
                continue  # Skip lines until reaching the desired start_line
            
            record = json.loads(line)  # Parse each line as JSON
            
            asn = record['asn']
            organization_id = record['organizationId']
            
            if organization_id not in org_id_map:
                org_id_map[organization_id] = []
            org_id_map[organization_id].append(asn)
    
    # Find the organizationId of the given ASN
    org_id_of_asn = None
    for org_id, asns in org_id_map.items():
        if asn_to_find in asns:
            org_id_of_asn = org_id
            break

    # If ASN is not found, return an empty list
    if org_id_of_asn is None:
        return []

    # Return all ASNs that share the same organizationId, excluding the given ASN
    siblings = [asn for asn in org_id_map[org_id_of_asn] if asn != asn_to_find]
    return siblings

# Test the function with a .jsonl.gz file, starting from line 10
# asn_to_find = "23752"
# siblings = find_siblings(asn_to_find)
# print(f"Siblings of ASN {asn_to_find}: {siblings}")
# Siblings of ASN 23752: ['55726', '58413']

In [ ]:
# Find number of IRR invalid prefixes looking into RADB
import subprocess

# ASN to exclude
excluded_asn = "AS32787"

# Files to store results
no_asn_32787_file = "no_asn_32787.txt" # Prefixes without ASN 32787
multiple_asns_file = "multiple_asns.txt" # Prefixes containing multiple other ASNs including ASN 32787

# Open output files
with open(no_asn_32787_file, "w") as no_asn_file, open(multiple_asns_file, "w") as multi_asn_file:
    for prefix in prefixes:
        # Run whois query and capture output
        result = subprocess.run(["whois", "-h", "whois.radb.net", prefix], capture_output=True, text=True)
        
        # Extract origin ASNs from whois output
        origin_asns = set()
        for line in result.stdout.splitlines():
            if line.lower().startswith("origin"):
                asn = line.split()[-1]
                origin_asns.add(asn)

        # Check conditions and write to respective files
        if excluded_asn not in origin_asns:
            no_asn_file.write(f"{prefix} - Origin ASNs: {', '.join(origin_asns)}\n")
        
        if len(origin_asns) > 1:
            multi_asn_file.write(f"{prefix} - Multiple Origin ASNs: {', '.join(origin_asns)}\n")

print(f"Prefixes without ASN {excluded_asn} have been saved to {no_asn_32787_file}")
print(f"Prefixes with multiple origin ASNs have been saved to {multiple_asns_file}")

In [ ]:
# Program to find upstream provider using condition: 
# a. If the same origin AS is prepending, upstream AS is the next one after prepending. 
# b. If the origin AS has siblings, remove siblings. 
# CAVEAT: This program does not check as_path = [200, 300, 400, 400, 8074, 8075, 8075] where there is a repeatation of 
# new ASes other than siblings. E.g. AS400 is repeated here.

# Function to find the immediate provider ASN
def find_immediate_provider(as_path):
    if len(as_path) < 2:
        return None  # Not enough ASNs in path to determine provider

    last_origin_asn = as_path[-1]  # The last ASN is the origin ASN

    # Check for sequentially repeated ASNs
    repeated_asn = None
    for i in range(len(as_path) - 1, 0, -1):
        if as_path[i-1] == as_path[i]:
            repeated_asn = as_path[i]
        else:
            # If we find an ASN that is not the same as the repeated ASN,
            # and we have found a repeated ASN, return the one before the repeated ASN
            if repeated_asn is not None:
                return as_path[i-1]  # This is the upstream provider
            break

    # Do this check to call API only in case an origin has mutliple siblings

    # If no repeated ASN found or it's the only ASN in the path
    if repeated_asn is None:
        # Retrieve siblings for the last ASN (last origin in the AS path)
        try:
            sibling_list = find_siblings(last_origin_asn)
        except Exception as e:
            print(f"Error retrieving siblings: {e}")
            return None  # Handle API failure appropriately

        # Traverse the AS path 
        for i in range(len(as_path) - 1, 0, -1):
            if str(as_path[i-1]) not in sibling_list:
                return as_path[i-1]  # The first non-sibling ASN is the upstream provider

    return as_path[1]  # If all checks fail, return the second ASN by default


# Test cases based on your examples
# as_path = [400, 100, 300, 300]  # Case 1: No repetition, no siblings
# as_path = ["200", "300", "400", "400", "6584", "8074", "8075"]  # Case 2: Sequentially repeated, should return 300


# # # Finding immediate providers
# print("Immediate provider for AS path:", find_immediate_provider(as_path))  # Expected Output: 300


In [22]:
# Program to find siblings of an ASN based on CAIDA AS2Org data
# Read line from 95721 from the file /h # format:aut|changed|aut_name|org_id|opaque_id|source
import gzip
import json

# Function to find siblings of a given ASN in a .jsonl.gz file, starting from a specific line
def find_siblings(asn_to_find):
    # Create a dictionary to group ASNs by organizationId
    org_id_map = {}
    
    file_path = '/home/shyam/jupy/ddos_scrubber/data/20241001.as-org2info.jsonl.gz'

    start_line = 95721
        
    # Open and read the .jsonl.gz file
    with gzip.open(file_path, 'rt') as f:  # 'rt' is for reading text mode
        for current_line, line in enumerate(f):
            if current_line < start_line:
                continue  # Skip lines until reaching the desired start_line
            
            record = json.loads(line)  # Parse each line as JSON
            
            asn = record['asn']
            organization_id = record['organizationId']
            
            if organization_id not in org_id_map:
                org_id_map[organization_id] = []
            org_id_map[organization_id].append(asn)
    
    # Find the organizationId of the given ASN
    org_id_of_asn = None
    for org_id, asns in org_id_map.items():
        if asn_to_find in asns:
            org_id_of_asn = org_id
            break

    # If ASN is not found, return an empty list
    if org_id_of_asn is None:
        return []

    # Return all ASNs that share the same organizationId, excluding the given ASN
    siblings = [asn for asn in org_id_map[org_id_of_asn] if asn != asn_to_find]
    return siblings

# Test the function with a .jsonl.gz file, starting from line 10
asn_to_find = "23752"
siblings = find_siblings(asn_to_find)
print(f"Siblings of ASN {asn_to_find}: {siblings}")
# Siblings of ASN 23752: ['55726', '58413']

Siblings of ASN 23752: ['55726', '58413']


In [1]:
# Find number of excluded customer inference.
# Remember: excluded inference are those AS paths where an scrubber ASN does not come
# a. immediately before the origin ASN
# b. before path prepending of the origin ASN
# c. before siblings (Inferred by CAIDA AS2Org dataset) of the origin ASN
# Example: 1798 3356 198949 56460 56460 50063
asn = "198949"
year = "2024"
import pandas as pd
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+"_v3.csv")
df2 = df.loc[df["siblings"] == 0] # Gives such inferred records

# Among the inferred records find the records whose inferences we were not able to find.

# Function to extract AS path after 198949
def extract_after_scrubber(as_path):
    parts = as_path.split()  # Split the AS path into individual ASNs
    if asn in parts:
        index = parts.index(asn)  # Find the index of 198949
        return " ".join(parts[index + 1:])  # Return the path after 198949
    return None  # Return None if 198949 is not present

# Apply function to extract AS paths after 198949
df2['as_path_after_scrubber'] = df2['as_path'].apply(extract_after_scrubber, axis=1)

# Find unique AS paths
unique_as_paths = df2['as_path_after_scrubber'].dropna().unique()

# Display results
print("\nNumber of unique AS paths after scrubber %s are: %s" %(asn, len(unique_as_paths)))


TypeError: extract_after_scrubber() got an unexpected keyword argument 'axis'